<a href="https://colab.research.google.com/github/sereenajoshy/AI-ML-Intership/blob/main/DAY%207/DAY_7_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/IMDB Dataset.csv', encoding = 'latin1')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [2]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [21]:
negation_words=[
    "not good","not bad","not great","don't like",
    "don't like","never liked","wasn't goog",
    "isn't good","no good"
]

def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']"," ",text)

  for phrase in negation_words:
    text=text.replace(phrase, phrase.replace(" ","_"))
  return text

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2
)

In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

#oov out of vocabulary
tokenizer=Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

x_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
x_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.5),
    Dense(1,activation='sigmoid')
])



In [26]:
model.compile(
    optimizer  = 'adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
history=model.fit(
    x_train_pad,
    y_train,
    batch_size=128,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 225s 889ms/step - accuracy: 0.5443 - loss: 0.6776 - val_accuracy: 0.5874 - val_loss: 0.6383
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 218s 872ms/step - accuracy: 0.6039 - loss: 0.6108 - val_accuracy: 0.6019 - val_loss: 0.6235
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 218s 871ms/step - accuracy: 0.6461 - loss: 0.5704 - val_accuracy: 0.5956 - val_loss: 0.6467
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 218s 872ms/step - accuracy: 0.6975 - loss: 0.5268 - val_accuracy: 0.8244 - val_loss: 0.5437
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 218s 871ms/step - accuracy: 0.8206 - loss: 0.4263 - val_accuracy: 0.8319 - val_loss: 0.4822


In [28]:
loss, acc=model.evaluate(x_test_pad,y_test)
print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 55ms/step - accuracy: 0.8269 - loss: 0.4801
Test Accuracy: 0.8269000053405762


In [40]:
def predict_sentiment(review):
  review = clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded_sequence=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded_sequence)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)

  if prediction >= 0.5:
    print("Sentiment: Positive")
  else:
    print("Sentiment: Negative")

In [41]:
predict_sentiment("This movie was absolutely amazing and I loved")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step

Review: this movie was absolutely amazing and i loved
Score: 0.7295928
Sentiment: Positive


In [42]:
predict_sentiment("This movie is not good")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

Review: this movie is not_good
Score: 0.29591894
Sentiment: Negative
